In [ ]:
"""
Linear Regression Analysis

Purpose
-------
Provides a robust, single-variable linear regression analysis tool 
built on top of pandas and scikit-learn. The class handles data verification, 
automated missing value removal, IQR-based outlier filtering, and provides 
an accessible summary of the resulting statistical model.
"""

# -------------------------------------------------
#                   Classes
# -------------------------------------------------

class LinearRegressionAnalysis():
    """
    A class to perform single-variable linear regression on pandas Series.

    This class automates data validation, pre-processing (including missing 
    value removal and IQR-based outlier detection), and fits a linear 
    regression model to compute the goodness of fit, slope, and intercept.

    Parameters
    ----------
    x : pd.Series
        The independent variable (feature) dataset.
    y : pd.Series
        The dependent variable (target) dataset.

    Attributes
    ----------
    x : pd.Series
        The raw independent variable data passed during initialization.
    y : pd.Series
        The raw dependent variable data passed during initialization.
    data : pd.DataFrame
        The cleaned, aligned DataFrame containing both `x` and `y` after 
        dropping NaNs and removing outliers.
    score : float
        The coefficient of determination (:math:`R^2`) of the prediction.
    coeff : np.ndarray
        The estimated linear regression coefficients (slope).
    inter : float
        The independent term (intercept) in the linear model.

    Raises
    ------
    ValueError
        If the lengths of `x` and `y` do not match.
    ValueError
        If the indices of `x` and `y` are not identical.

    Examples
    --------
    >>> import pandas as pd
    >>> idx = pd.date_range("2026-01-01", periods=5)
    >>> x_data = pd.Series([1, 2, 3, 4, 100], index=idx)  # 100 is an outlier
    >>> y_data = pd.Series([2, 4, 6, 8, 20], index=idx)
    >>> model = LinearRegressionAnalysis(x_data, y_data)
    >>> model.summary()
    ========================================
         LINEAR REGRESSION SUMMARY     
    ========================================
    R² Score (Goodness of Fit): 1.0000
    Slope (Coefficient):        2.0000
    Intercept:                  0.0000
    ----------------------------------------
    Mathematical Equation:
    y = (2.0000 * x) + (0.0000)
    ========================================
    """

    def __init__(self, x: pd.Series, y: pd.Series) -> None:
        self.x = x
        self.y = y
        self.check_length()
        self.check_index_alignment()
        self.data = self.combine_data()
        self.clean_data()
        self.score, self.coeff, self.inter = self.regression_analysis()

    def check_length(self) -> None:
        """
        Verify that the independent and dependent variables have equal lengths.

        Raises
        ------
        ValueError
            If ``len(self.x) != len(self.y)``.
        """
        if len(self.x) != len(self.y):
            raise ValueError('The length of x and y are not the same')
    
    def check_index_alignment(self) -> None:
        """
        Verify that the independent and dependent variables share the same index.

        Raises
        ------
        ValueError
            If the index elements or order differ between `x` and `y`.
        """
        if not self.x.index.equals(self.y.index):
            raise ValueError('The index of x and y are not the same')
    
    def combine_data(self) -> pd.DataFrame:
        """
        Concatenate the x and y Series into a single DataFrame along columns.

        Returns
        -------
        pd.DataFrame
            A combined DataFrame with `x` as the first column and `y` as 
            the second column.
        """
        return pd.concat([self.x, self.y], axis=1)
    
    def clean_data(self) -> None:
        """
        Clean the data by removing missing values and statistical outliers.

        This method drops rows with missing values (NaNs) and applies the Interquartile 
        Range (IQR) method to filter out statistical outliers. An observation is 
        considered an outlier if it falls outside:
        :math:`[Q1 - 1.5 \\times IQR, Q3 + 1.5 \\times IQR]`.

        Modifies
        --------
        self.data : pd.DataFrame
            Replaces the raw combined data with the filtered DataFrame.
        """
        data = self.data.copy()
        data = data.dropna()
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        upper_bound = q3 + (1.5 * iqr)
        data = data[~(((data < lower_bound) | (data > upper_bound)).any(axis=1))]
        self.data = data

    def regression_analysis(self):
        """
        Fit a ordinary least squares linear regression model on the cleaned data.

        Returns
        -------
        score : float
            The :math:`R^2` score of the model.
        coef_ : np.ndarray
            Array of estimated coefficients (slope).
        intercept_ : float
            The independent mathematical intercept term.
        """
        X = self.data.iloc[:, 0].values.reshape(-1, 1)
        y = self.data.iloc[:, 1].values
        reg = LinearRegression().fit(X, y)
        return reg.score(X, y), reg.coef_, reg.intercept_

    def summary(self) -> None:
        """
        Print a clean report of the regression analysis results.

        Outputs the calculated goodness of fit (R²), slope coefficient, 
        y-intercept, and the formatted deterministic algebraic equation.
        """
        print("=" * 40)
        print("     LINEAR REGRESSION SUMMARY     ")
        print("=" * 40)
        print(f"R² Score (Goodness of Fit): {self.score:.4f}")
        print(f"Slope (Coefficient):        {self.coeff[0]:.4f}")
        print(f"Intercept:                  {self.inter:.4f}")
        print("-" * 40)
        print(f"Mathematical Equation:")
        print(f"y = ({self.coeff[0]:.4f} * x) + ({self.inter:.4f})")
        print("=" * 40)
